# Python `str.partition()`: Master 3-Tuple Splitting & Safe Unpacking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_partition_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.partition() Tutorial](https://www.plus2net.com/python/string-partition.php)**.

### Learning Objectives:
1. Understand the guaranteed 3-tuple `(head, sep, tail)` return signature.
2. Prevent runtime crashes by replacing fragile `split(..., 1)` unpacking with safe `partition()`.
3. Contrast left-to-right `partition()` with right-to-left `rpartition()`.
4. Build production string parsers for configuration settings, URI schemes, and email domains.
5. Validate implementations with automated self-checking unit assertions.

## 1. 3-Tuple Return Mechanics & Guaranteed Safe Unpacking

`str.partition(sep)` splits at the first occurrence of `sep` from the left:
- If `sep` is found: returns `(head, sep, tail)`
- If `sep` is NOT found: returns `(original_string, "", "")`

In [ ]:
sentence = "Python is powerful and versatile"

# 1. Separator is present
head, sep, tail = sentence.partition("is")
print("When found:  ", (head, sep, tail))
assert head == "Python "
assert sep == "is"
assert tail == " powerful and versatile"

# 2. Separator is missing
m_head, m_sep, m_tail = sentence.partition("Java")
print("When missing:", (m_head, m_sep, m_tail))
assert m_head == sentence
assert m_sep == ""
assert m_tail == ""

# 3. Easy boolean verification
assert bool(sep) is True
assert bool(m_sep) is False

print("\nAll basic partition assertions passed!")

## 2. Crash Prevention: `split(..., 1)` vs. `partition(...)`

Attempting to unpack `split(sep, 1)` into two variables crashes if `sep` is not found. `partition()` never fails on unpack because its size is constant (always 3 items).

In [ ]:
bad_input = "DATABASE_NAME"  # Missing '='

# 1. split() crashes
try:
    k, v = bad_input.split("=", 1)
    assert False, "Expected ValueError not raised"
except ValueError as e:
    print("split() caught expected error:", e)

# 2. partition() unpacks safely
k, sep, v = bad_input.partition("=")
print("partition() safe unpack:      ", k, "|", sep, "|", v)
assert k == "DATABASE_NAME"
assert sep == ""
assert v == ""

print("\nSafe unpacking assertions passed!")

## 3. Directional Partitioning: `partition()` vs. `rpartition()`

Observe how the fallback tuple differs when the separator is absent:
- `partition()` returns `(original, "", "")`
- `rpartition()` returns `("", "", original)`

In [ ]:
filepath = "var/log/nginx/access.log"

# Left partition isolates first path component
root, sep_left, rest = filepath.partition("/")
print("partition(): ", (root, sep_left, rest))
assert root == "var"
assert rest == "log/nginx/access.log"

# Right partition isolates the leaf filename
dir_path, sep_right, filename = filepath.rpartition("/")
print("rpartition():", (dir_path, sep_right, filename))
assert dir_path == "var/log/nginx"
assert filename == "access.log"

# Missing separator comparison
raw = "standalone"
assert raw.partition(":") == ("standalone", "", "")
assert raw.rpartition(":") == ("", "", "standalone")

print("\nDirectional partitioning assertions passed!")

## 4. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.partition()`:
1. Resilient Configuration Parser with Default Values
2. URI Scheme & Authority Extractor
3. Safe Email Address Decomposition

In [ ]:
# Recipe 1: Resilient config parser
def parse_config(line: str, default_val: str = "true") -> tuple[str, str]:
    key, sep, val = line.partition("=")
    key = key.strip()
    val = val.strip() if sep else default_val
    return key, val

assert parse_config("CACHE = false") == ("CACHE", "false")
assert parse_config("DEBUG_MODE") == ("DEBUG_MODE", "true")

# Recipe 2: URI Scheme extractor
def extract_scheme(uri: str) -> tuple[str, str]:
    scheme, sep, remainder = uri.partition("://")
    return (scheme.lower(), remainder) if sep else ("http", uri)

assert extract_scheme("https://plus2net.com/docs") == ("https", "plus2net.com/docs")
assert extract_scheme("localhost:8080") == ("http", "localhost:8080")

# Recipe 3: Safe Email Parser
def parse_email(email: str) -> tuple[str, str] | None:
    mailbox, sep, domain = email.strip().partition("@")
    if not sep or not mailbox or not domain or "@" in domain:
        return None
    return mailbox, domain

assert parse_email("admin@plus2net.com") == ("admin", "plus2net.com")
assert parse_email("invalid_email_string") is None

print("All engineering recipes passed!")

## 5. Production Implementation & Self-Checking Unit Assertions

This production-grade `TuplePartitioner` class encapsulates HTTP header parsing, URL path/query extraction, and delimiter verification.

In [ ]:
class TuplePartitioner:
    @staticmethod
    def split_header(header_line: str) -> tuple[str, str]:
        """Extracts HTTP header name and value safely."""
        name, sep, val = header_line.partition(":")
        return name.strip(), val.strip()

    @staticmethod
    def split_path_query(url_path: str) -> tuple[str, str]:
        """Separates path endpoint from query string."""
        path, sep, query = url_path.partition("?")
        return path, query

    @staticmethod
    def has_delimiter(text: str, delimiter: str) -> bool:
        """Checks delimiter presence using partition tuple center."""
        _, sep, _ = text.partition(delimiter)
        return bool(sep)


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
partitioner = TuplePartitioner()

# 1. Unpacking header
name, val = partitioner.split_header("Content-Type: application/json; charset=utf-8")
assert name == "Content-Type"
assert val == "application/json; charset=utf-8"

# 2. Path & Query Splitting
path, query = partitioner.split_path_query("/api/v1/search?term=python&limit=10")
assert path == "/api/v1/search"
assert query == "term=python&limit=10"

# 3. Delimiter presence
assert partitioner.has_delimiter("KEY=VAL", "=") is True
assert partitioner.has_delimiter("NO_DELIM", "=") is False

# 4. Empty separator raises ValueError
try:
    "test".partition("")
    assert False, "Expected ValueError not raised"
except ValueError as e:
    assert "empty separator" in str(e)

print("All str.partition() production unit assertions passed successfully!")